# RL Maintenance Policy: DQN vs Rule-Based on CMAPSS FD001
DQN learns when to replace an engine given predicted RUL. Compared against a fixed threshold policy.

**Dependencies not in base Colab:**
```
stable-baselines3
gymnasium
```
Run the cell below once per runtime.

In [ ]:
%pip install -q stable-baselines3 gymnasium

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

## 1. Environment

In [ ]:
%%writefile /content/src/rl_env.py
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class MaintenanceEnv(gym.Env):
    """
    One episode = one engine lifetime.
    State:   predicted RUL at the current cycle (single float).
    Actions: 0 = do nothing, 1 = replace.
    Rewards encode operational cost: late replacement is penalised harder
    than early replacement, and failure without replacement is catastrophic.
    """

    def __init__(self, episodes):
        # episodes: list of dicts with 'pred_rul' and 'actual_rul' (1-D float32 arrays)
        super().__init__()
        self.episodes = episodes
        self.observation_space = spaces.Box(
            low=np.float32(0.0),
            high=np.float32(500.0),
            shape=(1,),
            dtype=np.float32,
        )
        self.action_space = spaces.Discrete(2)
        self._ep   = None
        self._step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # random sampling so the agent sees varied degradation trajectories
        idx        = int(self.np_random.integers(0, len(self.episodes)))
        self._ep   = self.episodes[idx]
        self._step = 0
        return np.array([self._ep['pred_rul'][0]], dtype=np.float32), {}

    def step(self, action):
        pred_rul   = float(self._ep['pred_rul'][self._step])
        actual_rul = float(self._ep['actual_rul'][self._step])
        terminated = False
        truncated  = False

        if action == 1:
            # reward boundary at 30 cycles: close enough to failure to be intentional,
            # far enough to avoid an emergency
            reward     = 20.0 if actual_rul <= 50 else -5
            terminated = True
        else:
            if actual_rul <= 0:
                reward     = -100.0
                terminated = True
            else:
                reward      = 1.0
                self._step += 1
                if self._step >= len(self._ep['pred_rul']):
                    # safety net: real CMAPSS episodes always end on actual_rul == 0
                    terminated = True

        next_pred = 0.0 if terminated else float(self._ep['pred_rul'][self._step])
        return np.array([next_pred], dtype=np.float32), reward, terminated, truncated, {}

In [ ]:
sys.path.insert(0, '/content/src')
from rl_env import MaintenanceEnv

## 2. Load data and define split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/src', exist_ok=True)

DRIVE_ROOT = '/content/drive/MyDrive/engine-data-project'
DEVICE     = torch.device('cpu')  # DQN at this scale does not need GPU

df = pd.read_parquet(f'{DRIVE_ROOT}/train_FD001_clean.parquet')
sensor_cols = [c for c in df.columns if c.startswith('sensor_')]

engine_ids  = sorted(df['engine_id'].unique())
n_val       = max(1, int(len(engine_ids) * 0.2))
val_engines = set(engine_ids[-n_val:])
trn_engines = set(engine_ids) - val_engines

print(f'train: {len(trn_engines)} engines  val: {len(val_engines)} engines')

## 3. LSTM state signal for validation engines

In [ ]:
WINDOW_LSTM = 30
RUL_CAP     = 125


class LSTMRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, 64, batch_first=True)
        self.drop1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(64, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.2)
        self.fc    = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out    = self.drop1(out)
        out, _ = self.lstm2(out)
        out    = self.drop2(out[:, -1, :])
        return self.fc(out).squeeze(1)


lstm = LSTMRegressor(len(sensor_cols)).to(DEVICE)
lstm.load_state_dict(torch.load(f'{DRIVE_ROOT}/lstm_fd001.pth', map_location=DEVICE))
lstm.eval()
print('LSTM weights loaded')

In [ ]:
lstm_df = df.copy()
lstm_df['rul'] = lstm_df['rul'].clip(upper=RUL_CAP)

windows, win_eids = [], []
for eid, group in lstm_df[lstm_df['engine_id'].isin(val_engines)].groupby('engine_id'):
    group = group.sort_values('cycle')
    s     = group[sensor_cols].values
    for i in range(len(group) - WINDOW_LSTM + 1):
        windows.append(s[i : i + WINDOW_LSTM])
        win_eids.append(eid)

X_val   = np.array(windows, dtype=np.float32)
win_eids = np.array(win_eids)


class ArrayDataset(Dataset):
    def __init__(self, X): self.X = torch.from_numpy(X)
    def __len__(self):     return len(self.X)
    def __getitem__(self, i): return self.X[i]


loader = DataLoader(ArrayDataset(X_val), batch_size=512, shuffle=False)
all_preds = []
with torch.no_grad():
    for xb in loader:
        all_preds.append(lstm(xb.to(DEVICE)).cpu().numpy())
lstm_preds = np.concatenate(all_preds)

print(f'LSTM inference done: {lstm_preds.shape[0]} predictions across {len(val_engines)} val engines')

In [ ]:
# pair LSTM predictions with actual RUL; windows start at cycle index WINDOW_LSTM-1
val_episodes = []
for eid in sorted(val_engines):
    eng        = df[df['engine_id'] == eid].sort_values('cycle')
    actual_rul = eng['rul'].values.astype(np.float32)
    cycles     = eng['cycle'].values
    pred_rul   = lstm_preds[win_eids == eid]
    start      = WINDOW_LSTM - 1  # first cycle the LSTM has a full window for
    val_episodes.append({
        'engine_id': eid,
        'pred_rul':  pred_rul,
        'actual_rul': actual_rul[start:],
        'cycles':    cycles[start:],
    })

print(f'val episodes built: {len(val_episodes)}')

## 4. Training episodes

In [ ]:
# train on LSTM predictions so the agent sees the same noisy signal it will face at eval time
trn_windows, trn_win_eids = [], []
for eid, group in lstm_df[lstm_df['engine_id'].isin(trn_engines)].groupby('engine_id'):
    group = group.sort_values('cycle')
    s     = group[sensor_cols].values
    for i in range(len(group) - WINDOW_LSTM + 1):
        trn_windows.append(s[i : i + WINDOW_LSTM])
        trn_win_eids.append(eid)

X_trn        = np.array(trn_windows, dtype=np.float32)
trn_win_eids = np.array(trn_win_eids)

trn_loader = DataLoader(ArrayDataset(X_trn), batch_size=512, shuffle=False)
trn_preds_list = []
with torch.no_grad():
    for xb in trn_loader:
        trn_preds_list.append(lstm(xb.to(DEVICE)).cpu().numpy())
trn_lstm_preds = np.concatenate(trn_preds_list)

train_episodes = []
for eid in sorted(trn_engines):
    eng        = df[df['engine_id'] == eid].sort_values('cycle')
    actual_rul = eng['rul'].values.astype(np.float32)
    pred_rul   = trn_lstm_preds[trn_win_eids == eid]
    start      = WINDOW_LSTM - 1
    train_episodes.append({
        'pred_rul':   pred_rul,
        'actual_rul': actual_rul[start:],
        'cycles':     eng['cycle'].values[start:],
    })

print(f'train episodes built: {len(train_episodes)}')

## 5. Train DQN

In [ ]:
train_env = Monitor(MaintenanceEnv(train_episodes))

dqn = DQN(
    'MlpPolicy',
    train_env,
    verbose=1,
    seed=42,
    exploration_fraction=0.3,
)
dqn.learn(total_timesteps=500_000)
print('DQN training complete')

## 6. Rule-based baseline

In [ ]:
# threshold of 20 gives the agent a small buffer before the reward boundary at 30
def rule_policy(pred_rul):
    return 1 if pred_rul <= 20 else 0


def dqn_policy(pred_rul):
    obs = np.array([pred_rul], dtype=np.float32)
    action, _ = dqn.predict(obs, deterministic=True)
    return int(action)

## 7. Evaluate both policies

In [ ]:
def run_episodes(policy_fn, episodes):
    rewards, failures, rep_cycles = [], [], []
    for ep in episodes:
        total     = 0
        failed    = False
        rep_cycle = None
        for t in range(len(ep['pred_rul'])):
            action = policy_fn(ep['pred_rul'][t])
            actual = ep['actual_rul'][t]
            if action == 1:
                total    += 20 if actual <= 30 else -10
                rep_cycle = ep['cycles'][t]
                break
            else:
                if actual <= 0:
                    total  -= 100
                    failed  = True
                    break
                total += 1
        else:
            # loop ended without break: no replacement and no recorded failure
            total  -= 100
            failed  = True
        rewards.append(total)
        failures.append(failed)
        rep_cycles.append(rep_cycle)
    return rewards, failures, rep_cycles


rule_rewards, rule_failures, rule_rep = run_episodes(rule_policy, val_episodes)
dqn_rewards,  dqn_failures,  dqn_rep  = run_episodes(dqn_policy,  val_episodes)

print('evaluation done')

## 8. Cumulative reward over episodes

In [ ]:
episodes_x = range(1, len(val_episodes) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(episodes_x, np.cumsum(rule_rewards), label='Rule-based')
ax.plot(episodes_x, np.cumsum(dqn_rewards),  label='DQN')
ax.set_xlabel('episode (validation engine)')
ax.set_ylabel('cumulative reward')
ax.set_title('Cumulative reward over validation episodes')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Results table

In [ ]:
def avg_or_na(vals):
    clean = [v for v in vals if v is not None]
    return round(np.mean(clean), 1) if clean else float('nan')


results = pd.DataFrame([
    {
        'Policy':                 'Rule-based',
        'Avg Reward':             round(np.mean(rule_rewards), 2),
        'Failure Rate':           f"{np.mean(rule_failures):.1%}",
        'Avg Replacement Cycle':  avg_or_na(rule_rep),
    },
    {
        'Policy':                 'DQN',
        'Avg Reward':             round(np.mean(dqn_rewards), 2),
        'Failure Rate':           f"{np.mean(dqn_failures):.1%}",
        'Avg Replacement Cycle':  avg_or_na(dqn_rep),
    },
])

print(results.to_string(index=False))